# 05 — Shapley / SAGE attribution: which feature and which agent carries the solves

Exact Shapley values: a player's credit is its average marginal contribution to the solve rate over every subset of
the other players. The **efficiency axiom** makes the split faithful — credits add up to the solve rate. Faithful to the
original `services/shapley.py`, `benchmarks/eval/feature_attribution.py` and `benchmarks/eval/sage.py`:

1. **Feature level (CV, free, label-free):** 11 feature families; v(S) = 1 if some measured property in S alone
   separates all 12 panels (the CV verifier). Per-problem credit, winning feature, dataset credit with a bootstrap CI,
   the diminishing-returns curve, and train/test transfer of the ranking.
2. **Agent level, proxy game (free):** each specialist is mapped to the feature families it reads (the original's
   `AGENT_FEATURES`; the relational agent is our addition).
3. **Agent level, graded game:** v(S) = share of problems where the highest-confidence round-1 rule among the
   specialists in S is Bongard's rule.
4. **CV verifier: single predicate vs conjunction** (the original's single-vs-conjunction experiment).

Monte-Carlo Shapley is included for games too large for exact enumeration and is cross-checked on the feature game.
The original's "real" agent coalitions (re-running the whole pipeline for every subset of agents — 64 runs per problem)
are not run here for cost; the proxy and graded games are the affordable counterparts.

In [ ]:
%run ./00_common.ipynb

In [ ]:
from scipy.stats import spearmanr
FEATURE_PROBLEMS = PROBLEM_SETS[os.environ.get("SAGE_FEATURE_SET", "ALL_100")]   # free: default all 100
PROBLEMS = PROBLEM_SETS[os.environ.get("PROBLEM_SET", "SMOKE")]
AGENT_ARM = "pipeline__specialist__gpt-oss-120b__ask_eye-judge"
rng = np.random.default_rng(0)

def shapley(players, v):
    """Exact Shapley values; weights s!(n-s-1)!/n!. Returns (phi, cached v)."""
    n, cache = len(players), {}
    def V(S):
        k = frozenset(S)
        if k not in cache:
            cache[k] = v(k)
        return cache[k]
    phi = {p: 0.0 for p in players}
    for p in players:
        others = [q for q in players if q != p]
        for r in range(len(others) + 1):
            w = math.factorial(r) * math.factorial(n - r - 1) / math.factorial(n)
            for sub in itertools.combinations(others, r):
                phi[p] += w * (V(set(sub) | {p}) - V(set(sub)))
    return phi, V

def shapley_mc(players, v, samples=1000, seed=0):
    """Monte-Carlo Shapley by random permutations (original shapley_values_mc)."""
    r, cache, phi = random.Random(seed), {}, {p: 0.0 for p in players}
    def V(S):
        k = frozenset(S)
        if k not in cache:
            cache[k] = v(k)
        return cache[k]
    for _ in range(samples):
        order = players[:]; r.shuffle(order)
        S = set()
        for p in order:
            before = V(S); S = S | {p}; phi[p] += V(S) - before
    return {p: x / samples for p, x in phi.items()}

def cumulative(players, phi, V):
    order = sorted(players, key=lambda p: -phi[p])
    return order, [V(set(order[:k])) for k in range(len(order) + 1)]

def plot_shapley(phi, V, players, title, fname, ci=None):
    order, curve = cumulative(players, phi, V)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.9))
    vals = [phi[p] * 100 for p in order][::-1]
    a1.barh(order[::-1], vals, color=C_OURS, height=0.6)
    if ci:
        lo = [(phi[p] - ci[p][0]) * 100 for p in order][::-1]; hi = [(ci[p][1] - phi[p]) * 100 for p in order][::-1]
        a1.errorbar(vals, range(len(vals)), xerr=[lo, hi], fmt="none", ecolor=C_INK2, elinewidth=1, capsize=3)
    for y, v in enumerate(vals):
        a1.text(max(v, 0) + 0.3, y + 0.25, f"{v:.1f}", va="center", fontsize=8.5, color=C_INK2)
    a1.set_xlabel("credit (percentage points)"); a1.set_title("credit per player (sums to the solve rate)", loc="left", fontsize=10)
    a2.plot(range(len(curve)), [c * 100 for c in curve], color=C_OURS, linewidth=2, marker="o", markersize=6)
    a2.set_xticks(range(len(curve))); a2.set_xticklabels(["none"] + order, rotation=40, ha="right", fontsize=8)
    a2.set_ylabel("problems solved (%)"); a2.set_title("adding players in order of credit (diminishing returns)", loc="left", fontsize=10)
    fig.suptitle(title, x=0.01, ha="left")
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS, "figures", fname), dpi=200); plt.show()

## 1. Feature level — 11 CV feature families (original `FEATURE_FAMILIES`)

In [ ]:
FAMILIES = {
    "size": ["area_frac", "large", "small"], "convexity": ["convexity"], "fill": ["fill_ratio", "outline", "solid"],
    "corners": ["ncorners", "triangle", "rectangle", "circle", "curved", "straight"], "compactness": ["compactness"],
    "elongation": ["elongation"], "orientation": ["aspect"], "position": ["centroid_x", "centroid_y"],
    "count": ["object_count"], "topology": ["inside", "overlap"], "symmetry": ["symmetric"],
}
FAM = list(FAMILIES)
solvable = {bp: verified_predicates(load_cv(bp)) for bp in FEATURE_PROBLEMS}

def v_problem(bp):
    return lambda S: 1.0 if solvable[bp] & {p for f in S for p in FAMILIES[f]} else 0.0

per_problem = {bp: shapley(FAM, v_problem(bp))[0] for bp in FEATURE_PROBLEMS}      # per-problem credit
phi_f = {f: float(np.mean([per_problem[bp][f] for bp in FEATURE_PROBLEMS])) for f in FAM}   # = dataset (SAGE) credit
def v_dataset(S):
    return float(np.mean([v_problem(bp)(S) for bp in FEATURE_PROBLEMS]))
_, V_f = shapley(FAM, v_dataset)
total = V_f(set(FAM))
boot = {f: [] for f in FAM}
for _ in range(2000):
    idx = rng.integers(0, len(FEATURE_PROBLEMS), len(FEATURE_PROBLEMS))
    for f in FAM:
        boot[f].append(np.mean([per_problem[FEATURE_PROBLEMS[i]][f] for i in idx]))
ci_f = {f: (float(np.percentile(boot[f], 2.5)), float(np.percentile(boot[f], 97.5))) for f in FAM}
winning = {bp: max(per_problem[bp], key=per_problem[bp].get) for bp in FEATURE_PROBLEMS if max(per_problem[bp].values()) > 1e-9}
print(f"{len(FEATURE_PROBLEMS)} problems | CV verifier solves {total:.0%} with all families | "
      f"efficiency gap {abs(sum(phi_f.values()) - total):.1e}")
print("winning feature (problems it decides):", dict(pd.Series(winning).value_counts()))
mc = shapley_mc(FAM, v_dataset, samples=400)
print(f"Monte-Carlo vs exact, max difference: {max(abs(mc[f] - phi_f[f]) for f in FAM):.4f}")
pd.DataFrame([{"family": f, "shapley": phi_f[f], "ci_low": ci_f[f][0], "ci_high": ci_f[f][1]} for f in FAM]) \
  .sort_values("shapley", ascending=False).to_csv(os.path.join(RESULTS, "tables", "shapley_features.csv"), index=False)
pd.DataFrame([{"bp": bp, "winning_feature": winning.get(bp, "")} for bp in FEATURE_PROBLEMS]) \
  .to_csv(os.path.join(RESULTS, "tables", "winning_feature.csv"), index=False)
plot_shapley(phi_f, V_f, FAM, f"Feature-level SAGE (CV verifier), {len(FEATURE_PROBLEMS)} problems", "shapley_features.png", ci=ci_f)

### Train/test transfer — is the feature ranking stable on problems it was not computed on? (original `train_test_transfer`)

In [ ]:
half = len(FEATURE_PROBLEMS) // 2
train, test = FEATURE_PROBLEMS[:half], FEATURE_PROBLEMS[half:]
phi_tr = {f: np.mean([per_problem[bp][f] for bp in train]) for f in FAM}
phi_te = {f: np.mean([per_problem[bp][f] for bp in test]) for f in FAM}
rho = spearmanr([phi_tr[f] for f in FAM], [phi_te[f] for f in FAM]).correlation
v_test = lambda S: float(np.mean([v_problem(bp)(S) for bp in test]))
order = sorted(FAM, key=lambda f: -phi_tr[f])
full = v_test(set(FAM))
print(f"train {len(train)} / test {len(test)} problems | Spearman rho between rankings = {rho:.2f}")
for k in (1, 2, 3, 5):
    print(f"   top-{k} train features keep {v_test(set(order[:k])) / full:.0%} of the test solve rate" if full else "")

## 2. Agent level, proxy game — each specialist credited with the feature families it reads (free)

In [ ]:
AGENT_FEATURES = {"shape": ("convexity", "corners", "compactness"), "contour": ("convexity", "corners"),
                  "size": ("size", "elongation"), "spatial": ("position", "orientation", "elongation"),
                  "quantity": ("count",), "fill": ("fill",), "topology": ("topology",),
                  "gestalt": ("symmetry", "position"), "relation": ("size", "position")}                   # the families each specialist reads in 02
AG = list(AGENT_FEATURES)
v_agents_proxy = lambda S: v_dataset({f for a in S for f in AGENT_FEATURES[a]})
phi_ap, V_ap = shapley(AG, v_agents_proxy)
print(f"proxy game: all agents' families solve {V_ap(set(AG)):.0%} | efficiency gap {abs(sum(phi_ap.values()) - V_ap(set(AG))):.1e}")
pd.DataFrame({"agent": AG, "shapley": [phi_ap[a] for a in AG]}).sort_values("shapley", ascending=False) \
  .to_csv(os.path.join(RESULTS, "tables", "shapley_agents_proxy.csv"), index=False)
plot_shapley(phi_ap, V_ap, AG, "Agent-level SAGE, proxy game (feature families each agent reads)", "shapley_agents_proxy.png")

## 3. Agent level, graded game — whose round-1 rule is Bongard's?

In [ ]:
preds = load_predictions(AGENT_ARM)
bps = [bp for bp in PROBLEMS if bp in preds]
if not bps:
    print(f"no saved predictions for {AGENT_ARM} on this problem set — run 02_pipeline first")
else:
    METER.tag = "grading specialist rules"
    agents = list(preds[bps[0]]["round1"])
    correct = {bp: {a: bool(preds[bp]["round1"][a].get("has_rule") and grade(bp, preds[bp]["round1"][a]["rule"])) for a in agents} for bp in bps}
    conf = {bp: {a: preds[bp]["round1"][a]["confidence"] for a in agents} for bp in bps}
    def v_agents(S):
        if not S:
            return 0.0
        return float(np.mean([correct[bp][max(S, key=lambda a: conf[bp][a])] for bp in bps]))
    phi_a, V_a = shapley(agents, v_agents)
    total = V_a(set(agents))
    print(f"{len(bps)} problems | best-confidence specialist right {total:.0%} | "
          f"any specialist right {np.mean([any(correct[bp].values()) for bp in bps]):.0%} | "
          f"efficiency gap {abs(sum(phi_a.values()) - total):.1e}")
    pd.DataFrame({"agent": list(phi_a), "shapley": list(phi_a.values())}).sort_values("shapley", ascending=False) \
      .to_csv(os.path.join(RESULTS, "tables", "shapley_agents.csv"), index=False)
    plot_shapley(phi_a, V_a, agents, f"Agent-level SAGE, graded game ({AGENT_ARM}), {len(bps)} problems", "shapley_agents.png")

## 4. CV verifier: single predicate vs conjunction (original experiment; free)

In [ ]:
single = {bp: propose_verified_rule(load_cv(bp)) for bp in FEATURE_PROBLEMS}
conj = {bp: single[bp] or propose_verified_rule(load_cv(bp), allow_conjunction=True) for bp in FEATURE_PROBLEMS}
n_s, n_c = sum(bool(v) for v in single.values()), sum(bool(v) for v in conj.values())
print(f"CV verifier proves a rule on {n_s}/{len(FEATURE_PROBLEMS)} problems with single predicates, "
      f"{n_c}/{len(FEATURE_PROBLEMS)} allowing a conjunction of two (conjunctions over-fit: check them semantically)")
pd.DataFrame([{"bp": bp, "single": (single[bp] or {}).get("nl", ""), "with_conjunction": (conj[bp] or {}).get("nl", "")}
              for bp in FEATURE_PROBLEMS]).to_csv(os.path.join(RESULTS, "tables", "cv_single_vs_conjunction.csv"), index=False)
METER.report()